# Setup

In [1]:
! pip install -qqU transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import warnings
import os
warnings.simplefilter(action='ignore', category=FutureWarning)

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [3]:
class CFG:
    model = "microsoft/Phi-4-mini-instruct"
    device = "cpu"

if torch.backends.mps.is_available():
    CFG.device = "mps"
if torch.cuda.is_available():
    CFG.device = "cuda"


print("Device: ", CFG.device)

Device:  cuda


# Funkcje

In [4]:
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(CFG.model, quantization_config=quantization_config)

config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now default to True since model is quantized.


model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [5]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model)

tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/3.91M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

In [6]:
def generate_text(
    prompt,
    temperature,
    top_p,
    model=model,
    tokenizer=tokenizer,
    max_length=100,
    num_return_sequences=1,
):
    # Set up the text generation pipeline
    generator = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        torch_dtype=torch.float16,
        device_map=CFG.device,
    )

    # Generate text
    generated = generator(
        prompt,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        truncation=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    return generated[0]["generated_text"]

Na początku funkcja przyjmuje szereg parametrów, które kontrolują proces generowania tekstu:
- `prompt` to tekst wejściowy, od którego model zacznie generację
- `temperature` kontroluje losowość generacji - wyższa wartość daje bardziej kreatywne, ale mniej przewidywalne wyniki
- `top_p` określa próg prawdopodobieństwa dla wyboru następnego słowa (nazywany próbkowaniem nucleus)
- `model_name` to nazwa modelu, którego chcemy użyć
- `max_length` ustala maksymalną długość generowanego tekstu (domyślnie 100 tokenów)
- `num_return_sequences` określa liczbę różnych wersji tekstu do wygenerowania (domyślnie 1)

W pierwszym kroku funkcja tworzy obiekt `generator` używając funkcji `pipeline`. Ten generator jest konfigurowany następującymi parametrami:
- typ zadania ustawiony na 'text-generation'
- wskazany model językowy
- `torch_dtype=torch.float16` oznacza użycie 16-bitowej precyzji liczb zmiennoprzecinkowych, co oszczędza pamięć
- `device_map` wskazuje na urządzenie obliczeniowe zdefiniowane w klasie CFG

Następnie funkcja wywołuje generator z wcześniej ustalonymi parametrami. Dodatkowe parametry w tym wywołaniu to:
- `do_sample=True` włącza próbkowanie losowe zamiast wybierania zawsze najbardziej prawdopodobnego słowa
- `truncation=True` pozwala na przycięcie tekstu do zadanej długości
- `pad_token_id=None` wyłącza automatyczne dopełnianie sekwencji

Na końcu funkcja zwraca wygenerowany tekst, wybierając pierwszy element z listy wyników (indeks [0]) i pobierając z niego klucz 'generated_text'.

# Test

In [7]:
prompt = "In a world where technology and nature coexist,"

Te dwa parametry, temperatura i top_p, są kluczowe w procesie generacji tekstu:
- Temperatura działa jak "pokrętło kreatywności". Przy niskich wartościach model będzie bardziej konserwatywny i przewidywalny, wybierając najbardziej prawdopodobne słowa. Przy wysokich wartościach staje się bardziej twórczy i nieprzewidywalny.
- Top_p kontroluje różnorodność słownictwa poprzez ograniczenie puli słów, z których model może wybierać. Niższe wartości prowadzą do bardziej skoncentrowanego, spójnego tekstu, podczas gdy wyższe pozwalają na większą różnorodność językową.

In [8]:

# Test different temperature values
temperatures = [0.2, 0.5, 1.0, 1.5]
for temp in temperatures:
    print('------')
    print(f"\nTemperature: {temp}, Top_p: 1.0")
    generated_text = generate_text(prompt, model = model, temperature=temp, top_p=1.0)
    print(generated_text)

Device set to use cuda:0


------

Temperature: 0.2, Top_p: 1.0


Device set to use cuda:0


In a world where technology and nature coexist, a young inventor named Alex discovers a hidden valley filled with ancient trees that possess unique properties. Alex learns that these trees can communicate through a network of roots and leaves, forming a symbiotic relationship with the surrounding ecosystem. Fascinatingly, the trees can also absorb and store energy from the sun, wind, and even the vibrations of the earth. Alex decides to harness this energy and create a sustainable power source for the nearby village, which has been struggling with
------

Temperature: 0.5, Top_p: 1.0


Device set to use cuda:0


In a world where technology and nature coexist, the concept of 'digital roots' merges with the essence of natural growth, giving rise to a fascinating phenomenon known as 'Bio-Digital Growth Rings'. Imagine a scenario where trees, imbued with a form of consciousness, can communicate with our digital world. Your task is to write a code that simulates the growth rings of a tree in a digital environment, where each ring represents a significant event in the tree's life, akin to the rings of a
------

Temperature: 1.0, Top_p: 1.0


Device set to use cuda:0


In a world where technology and nature coexist, there's a renowned robotics company tasked with creating a highly innovative robot designed to assist in the preservation of endangered species. This robot must navigate a wide range of environments, from dense rainforests to arid desert landscapes, with seamless adaptability to the requirements of each ecosystem for conservation purposes. The robot, named EcoGuardian and nicknamed 'The Watchful Protector,' shall use advanced AI to learn, adapt, and interact as naturally as possible within these ecosystems to provide
------

Temperature: 1.5, Top_p: 1.0
In a world where technology and nature coexist, you've uncovered an anomaly - a virus with bizarre traits that are a paradox to the norms in this parallel reality. A highly contagious disease, it has the power to simultaneously heal wounds and cause blight - it thrives in blight-stalled growth but regenerates vegetation instantly, mirroring an intriguing contradiction. The affected victims

Ten fragment kodu demonstruje eksperyment z wpływem temperatury na generowanie tekstu, utrzymując stałą wartość parametru top_p.

Lista `temperatures = [0.2, 0.5, 1.0, 1.5]` definiuje cztery różne wartości temperatury, które będą testowane:
- 0.2 reprezentuje bardzo niską temperaturę, gdzie model będzie bardzo konserwatywny i przewidywalny
- 0.5 to umiarkowanie niska temperatura, dająca względnie spójne, ale nie całkiem sztywne wyniki
- 1.0 to standardowa temperatura, gdzie model ma równowagę między kreatywnością a spójnością
- 1.5 to wysoka temperatura, która pozwoli modelowi na bardziej nieprzewidywalne i kreatywne generacje

Utrzymanie stałej wartości top_p = 1.0 jest istotne metodologicznie - pozwala zobaczyć czysty wpływ temperatury na generację, bez zakłóceń wynikających ze zmian innych parametrów. To jak eksperyment naukowy, gdzie zmieniamy tylko jedną zmienną, podczas gdy pozostałe pozostają stałe.

Wartość top_p = 1.0 oznacza, że model może wybierać ze wszystkich dostępnych słów, ważąc je tylko według ich prawdopodobieństwa zmodyfikowanego przez temperaturę. Jest to najbardziej "otwarte" ustawienie parametru top_p, co pozwala na pełne zaobserwowanie wpływu temperatury.

In [9]:
# Test different top_p values
top_ps = [0.5, 0.7, 0.9, 1.0]
for top_p in top_ps:
    print('------')
    print(f"\nTemperature: 1.0, Top_p: {top_p}")
    generated_text = generate_text(prompt, model = model, temperature=1.0, top_p=top_p)
    print(generated_text)

Device set to use cuda:0


------

Temperature: 1.0, Top_p: 0.5


Device set to use cuda:0


In a world where technology and nature coexist, a group of scientists, engineers, and artists come together to create a groundbreaking invention. They have developed a device that can harness the power of the sun, wind, and water to generate clean energy. This invention has the potential to revolutionize the way we power our world, reducing our dependence on fossil fuels and combating climate change.

The team, led by Dr. Emily Thompson, a renowned scientist, has spent years perfecting their invention. They have conducted
------

Temperature: 1.0, Top_p: 0.7


Device set to use cuda:0


In a world where technology and nature coexist, the 'Xeno-Synergy System' has been introduced. This innovative system aims to harmonize the seemingly divergent realms of digital advancement and ecological preservation, promising a future where technology serves as a guardian of nature rather than its conqueror. The 'Xeno-Synergy System' is a multi-faceted approach that integrates advanced AI, IoT devices, and renewable energy sources to create a sustainable and symbiotic relationship between humanity and the environment. Below
------

Temperature: 1.0, Top_p: 0.9


Device set to use cuda:0


In a world where technology and nature coexist, a tech mogul dreams of creating an advanced greenhouse that not only fosters plant growth but also serves as a living example of ecological sustainability. His vision includes a state-of-the-art greenhouse complex that integrates cutting-edge technology with natural processes, aiming to revolutionize the way we think about urban agriculture and environmental stewardship.

The tech mogul envisions the Green Haven Complex, a sprawling ecosystem where high-tech innovation meets organic farming. The complex is divided into several zones,
------

Temperature: 1.0, Top_p: 1.0
In a world where technology and nature coexist, a young child, Alex, is given the task to create a balanced environment in a virtual garden simulation. In this simulation, Alex can plant trees, flowers, and create water bodies, all while maintaining a balance with the virtual creatures that inhabit the garden.

In the garden, there are:
- Type A virtual creatures that repr

Ten fragment kodu stanowi drugi etap eksperymentu, który bada wpływ parametru top_p na generowanie tekstu. Tym razem utrzymujemy stałą temperaturę (1.0), zmieniając wartość top_p.

Lista `top_ps = [0.5, 0.7, 0.9, 1.0]` zawiera cztery starannie dobrane wartości top_p. Parametr top_p, znany również jako próbkowanie nucleus, określa łączne prawdopodobieństwo słów, które model może wybrać w każdym kroku generacji. Przykładowo, gdy top_p = 0.7, model będzie wybierał tylko spośród słów, których skumulowane prawdopodobieństwa nie przekraczają 70% całkowitego rozkładu prawdopodobieństwa.

Wybrane wartości top_p mają następujące znaczenie:
- 0.5 oznacza bardzo selektywny wybór - model ogranicza się do najbardziej prawdopodobnych słów, co powinno prowadzić do bardziej przewidywalnego i zachowawczego tekstu
- 0.7 to umiarkowanie restrykcyjna wartość, dająca modelowi nieco więcej swobody, ale wciąż utrzymująca znaczącą kontrolę nad wyborem słów
- 0.9 pozwala na szerszy wybór słów, zbliżając się do pełnej swobody
- 1.0 daje modelowi dostęp do całego słownika, podobnie jak w poprzednim eksperymencie

Stała temperatura 1.0 została wybrana jako wartość neutralna - nie jest ani zbyt zachowawcza, ani zbyt ryzykowna. Dzięki temu możemy wyraźnie zaobserwować efekty zmiany parametru top_p.

Połączenie wyników obu eksperymentów (z różnymi temperaturami i różnymi wartościami top_p) pozwala zrozumieć, jak te dwa parametry współdziałają ze sobą w procesie generowania tekstu i jak można je dostosowywać do osiągnięcia pożądanych efektów.